# Task 3.3: Basic Convolutional Neural Network – Iteration 1

**Project:** Potato Quality Classification (Fresh vs. Rotten)  
**ILO:** 8.2 B & D  

This notebook covers the first full iteration of building a CNN for image classification:

- **A. Preprocessing** – resize images to 128×128 and normalise pixel values to [0, 1]
- **B. CNN Architecture** – Conv2D, BatchNorm, MaxPooling, Dropout, Dense layers
- **C. Training** – Adam optimiser with early stopping
- **D. Evaluation** – confusion matrix, accuracy, precision, recall, F1; comparison against baselines

---

**Baselines to beat:**
| Model | Accuracy |
|---|---|
| Random Guess | 50.18 % |
| MLP Baseline | 79.03 % |

## 1. Imports & Setup

We fix a global seed (42) for NumPy and TensorFlow so that results are reproducible across runs.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

## 2. Data Loading & Preprocessing

### Design choices
| Setting | Value | Rationale |
|---|---|---|
| Image size | 128 × 128 px | Good balance between spatial detail and memory usage |
| Normalisation | ÷ 255 → [0, 1] | Stable gradients; avoids double-normalisation |
| Batch size | 32 | Standard default; fits in memory |
| Class mode | categorical | Needed for softmax output with 2 classes |

The data are already split into **train / val / test** folders so we use `ImageDataGenerator` to stream them directly from disk without loading everything into RAM.

In [ ]:
# ── Paths (relative from this notebook) ──────────────────────────────────────
BASE_PATH  = '../Task 3.1 Create an Image Dataset ILO 6.2 a en b/data/processed'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE   = (128, 128)   # height × width
BATCH_SIZE = 32

# ── Data generators (rescale only – no augmentation in iteration 1) ───────────
datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen = datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())  # ['fresh', 'rotten']
print('Classes      :', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Val samples  :', val_gen.samples)
print('Test samples :', test_gen.samples)

In [ ]:
# ── Sanity check: visualise a batch ──────────────────────────────────────────
images, labels = next(train_gen)
print(f'Batch shape : {images.shape}')   # (32, 128, 128, 3)
print(f'Pixel range : [{images.min():.3f}, {images.max():.3f}]')  # should be [0, 1]

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, img, label in zip(axes.flat, images[:10], labels[:10]):
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[int(label.argmax())], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Training Images (normalised to [0, 1])', y=1.02)
plt.tight_layout()
plt.show()

## 3. CNN Architecture – cnn_v1

### Architecture rationale

The network consists of **three convolutional blocks** followed by a **fully-connected classifier head**.

| Block | Layers | Purpose |
|---|---|---|
| Block 1 | Conv2D(32) → BatchNorm → MaxPool → Dropout | Low-level edge & colour features |
| Block 2 | Conv2D(64) → BatchNorm → MaxPool → Dropout | Mid-level texture features |
| Block 3 | Conv2D(128) → BatchNorm → MaxPool → Dropout | High-level abstract features |
| Head | Flatten → Dense(256) → BatchNorm → Dropout → Dense(2) | Classification |

**Layer choices:**
- **Conv2D** with `padding='same'` preserves spatial dimensions across the layer.
- **BatchNormalization** after each Conv accelerates training and acts as a mild regulariser.
- **MaxPooling2D(2×2)** halves the spatial size, reducing parameters and adding translation invariance.
- **Dropout** (0.25 in conv blocks, 0.5 in the head) prevents co-adaptation of neurons (overfitting).
- **Softmax** output with `categorical_crossentropy` loss is standard for multi-class classification.

In [ ]:
def build_cnn_v1(input_shape=(128, 128, 3), n_classes=2):
    """First CNN iteration: 3 conv blocks + dense head."""
    model = Sequential([
        Input(shape=input_shape),

        # ── Convolutional Block 1 ─────────────────────────────────────────
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Convolutional Block 2 ─────────────────────────────────────────
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Convolutional Block 3 ─────────────────────────────────────────
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # ── Classifier Head ───────────────────────────────────────────────
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(n_classes, activation='softmax')
    ], name='cnn_v1')
    return model


cnn_v1 = build_cnn_v1()
cnn_v1.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_v1.summary()

## 4. Training with Early Stopping

**Early stopping** monitors `val_loss` and stops training when it no longer improves. Setting `patience=5` gives the model 5 consecutive epochs to recover before stopping. `restore_best_weights=True` ensures the saved model weights correspond to the epoch with the lowest validation loss, not the last epoch.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_v1 = cnn_v1.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTraining stopped after {len(history_v1.history["loss"])} epoch(s).')

## 5. Learning Curves

The learning curves show how training and validation **loss** and **accuracy** evolve over epochs. A well-behaved model shows both curves converging. A large gap between train and val curves indicates overfitting.

In [ ]:
def plot_learning_curves(history, title_suffix=''):
    """Plot loss and accuracy curves side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(history.history['loss'],     label='Train loss')
    axes[0].plot(history.history['val_loss'], label='Val loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Loss {title_suffix}')
    axes[0].legend()

    # Accuracy
    axes[1].plot(history.history['accuracy'],     label='Train accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'Accuracy {title_suffix}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_learning_curves(history_v1, title_suffix='– CNN v1')

## 6. Evaluation on the Test Set

We evaluate the trained model on the held-out test set using the following metrics:

| Metric | Why it matters |
|---|---|
| **Accuracy** | Overall fraction of correct predictions |
| **Precision** | Of all predicted positives, how many are correct? Relevant when false positives are costly |
| **Recall** | Of all actual positives, how many did we catch? Relevant when false negatives are costly |
| **F1 Score** | Harmonic mean of precision and recall; balances both |
| **Confusion Matrix** | Shows per-class error patterns |

We also plot the **confusion matrix** as a heatmap to visualise which classes are confused.

In [ ]:
# ── Predict on test set ───────────────────────────────────────────────────────
test_gen.reset()
y_pred_proba = cnn_v1.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)   # predicted class indices
y_true = test_gen.classes                   # ground-truth class indices

# ── Metrics ───────────────────────────────────────────────────────────────────
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted')
rec  = recall_score(y_true, y_pred, average='weighted')
f1   = f1_score(y_true, y_pred, average='weighted')

print('=' * 45)
print(f'  Accuracy  : {acc  * 100:.2f} %')
print(f'  Precision : {prec * 100:.2f} %')
print(f'  Recall    : {rec  * 100:.2f} %')
print(f'  F1 Score  : {f1   * 100:.2f} %')
print('=' * 45)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix – CNN v1 (Test Set)')
plt.tight_layout()
plt.show()

## 7. Baseline Comparison

We compare the CNN against the two baselines established in Task 3.2.

In [ ]:
RANDOM_GUESS_ACC = 50.18
MLP_BASELINE_ACC = 79.03
cnn_v1_acc       = acc * 100

models   = ['Random Guess', 'MLP Baseline', 'CNN v1']
acc_vals = [RANDOM_GUESS_ACC, MLP_BASELINE_ACC, cnn_v1_acc]
colours  = ['#d62728', '#ff7f0e', '#2ca02c']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, acc_vals, color=colours, edgecolor='black', width=0.5)

for bar, val in zip(bars, acc_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.2f} %',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison – Test Set Accuracy', fontsize=13)
ax.axhline(y=100, color='grey', linestyle='--', alpha=0.4, label='Perfect (100 %)')
ax.legend()
plt.tight_layout()
plt.show()

print('\n── Comparison Table ──────────────────────────────────')
print(f'  Random Guess : {RANDOM_GUESS_ACC:.2f} %  (baseline – coin flip)')
print(f'  MLP Baseline : {MLP_BASELINE_ACC:.2f} %  (flat network, no spatial features)')
print(f'  CNN v1       : {cnn_v1_acc:.2f} %  (convolutional network, iteration 1)')
delta_mlp = cnn_v1_acc - MLP_BASELINE_ACC
print(f'\n  Δ vs MLP     : {delta_mlp:+.2f} percentage points')

## 8. Summary & Conclusions Iteration 1

In this first iteration a complete CNN pipeline was built to classify potatoes as fresh or rotten. Images were resized to 128x128 and normalised to [0, 1]. The CNN (cnn_v1) uses three convolutional blocks (32 → 64 → 128 filters) each followed by BatchNorm, MaxPooling and Dropout, plus a Dense head (256 → 2 softmax). It was trained with Adam (lr=0.001) and early stopping (patience=5).

**Training** stopped after 24 epochs, restoring weights from epoch 19 (best val_loss = 0.2396). The validation loss was very unstable in the early epochs (starting at 10.86 in epoch 1), which is common when BatchNormalization needs a few epochs to stabilise its running statistics.

**Test set results:**
- Accuracy: 94.35%
- Precision: 94.40%
- Recall: 94.35%
- F1: 94.36%

The model correctly classified 352 out of 372 test images. Fresh had 9 misclassifications, rotten had 14.

**Baseline comparison:**
- Random Guess: 50.18%
- MLP Baseline: 79.03%
- CNN v1: 94.35% (+15.32 pp over MLP)

The CNN outperforms the MLP because convolutional filters capture spatial patterns like texture and discolouration that are characteristic of rot, whereas the MLP treats every pixel independently and loses all spatial information.

**Points to improve in iteration 2:**
- The unstable early val_loss suggests a lower learning rate or learning rate warm-up could help
- A small gap between train (~94%) and val (~92%) accuracy suggests mild overfitting — data augmentation could help
- The Dense layer after Flatten accounts for 8.4M of the 8.5M total parameters; replacing it with Global Average Pooling would make the model much lighter